# 01 · Cargar el Lakehouse de Banco Brétema

Este notebook crea las tablas Delta que usan la ontología, el agente de datos y la Fabric App.

1. Descarga los CSV sintéticos del repositorio de GitHub, o usa los que hayas subido a `Files/datos`.
2. Crea las ocho tablas del banco con tipos explícitos: oficinas, gestores, clientes, cuentas, tarjetas, comercios, transacciones y alertas.
3. Crea dos tablas preparadas para la Fabric App: `app_alertas` y `app_movimientos`.
4. Comprueba las cifras que usa el guion de la demo.

**Antes de ejecutarlo:** añade el Lakehouse `LH_Bretema` como Lakehouse predeterminado del notebook, en el panel **Explorador**, a la izquierda.

Los datos son ficticios. Banco Brétema no existe.

In [ ]:
# Parámetros
ORIGEN = "github"  # "github": descarga los CSV del repositorio · "files": usa los que ya estén en Files/datos
URL_DATOS = "https://raw.githubusercontent.com/AbenFdez/Banco_Bretema_Prevencion_Fraude/main/datos"
CARPETA = "Files/datos"

## 1. Traer los CSV a `Files/datos`

Con `ORIGEN = "github"` el notebook descarga los ocho ficheros (unos 14 MB). Si tu capacidad no tiene salida a internet, súbelos a mano a `Files/datos` desde el Lakehouse (**Obtener datos › Cargar archivos**) y cambia `ORIGEN` a `"files"`.

In [ ]:
import os
import urllib.request

TABLAS = ["oficinas", "gestores", "clientes", "cuentas", "tarjetas", "comercios", "transacciones", "alertas"]
ruta_local = f"/lakehouse/default/{CARPETA}"

if ORIGEN == "github":
    os.makedirs(ruta_local, exist_ok=True)
    for tabla in TABLAS:
        urllib.request.urlretrieve(f"{URL_DATOS}/{tabla}.csv", f"{ruta_local}/{tabla}.csv")
        print(f"Descargado {tabla}.csv")
else:
    faltan = [t for t in TABLAS if not os.path.exists(f"{ruta_local}/{t}.csv")]
    assert not faltan, f"Faltan estos CSV en {CARPETA}: {faltan}"
    print("Los ocho CSV están en", CARPETA)

## 2. Crear las tablas del banco

Los tipos se fijan a mano en lugar de inferirlos, porque la ontología solo admite claves de texto o enteras y el grafo no admite `DECIMAL`. Por eso los importes son `DOUBLE`. Las columnas clave se marcan como no nulas.

In [ ]:
ESQUEMAS = {
    "oficinas": "oficina_id STRING, nombre STRING, municipio STRING, provincia STRING, tipo STRING, latitud DOUBLE, longitud DOUBLE",
    "gestores": "gestor_id STRING, nombre STRING, rol STRING, oficina_id STRING",
    "clientes": "cliente_id STRING, nombre STRING, tipo_cliente STRING, segmento STRING, edad INT, municipio STRING, provincia STRING, fecha_alta DATE, oficina_id STRING, gestor_id STRING, riesgo_kyc STRING",
    "cuentas": "cuenta_id STRING, cliente_id STRING, tipo_cuenta STRING, fecha_apertura DATE, saldo_eur DOUBLE, estado STRING",
    "tarjetas": "tarjeta_id STRING, cuenta_id STRING, cliente_id STRING, tipo_tarjeta STRING, tarjeta_enmascarada STRING, fecha_emision DATE, fecha_caducidad DATE, limite_eur DOUBLE, estado STRING, fecha_bloqueo DATE",
    "comercios": "comercio_id STRING, nombre STRING, categoria STRING, municipio STRING, pais STRING, canal STRING, nivel_riesgo STRING, latitud DOUBLE, longitud DOUBLE",
    "transacciones": "transaccion_id STRING, tarjeta_id STRING, comercio_id STRING, fecha_hora TIMESTAMP, importe_eur DOUBLE, canal STRING, pais STRING, resultado STRING",
    "alertas": "alerta_id STRING, transaccion_id STRING, tarjeta_id STRING, cliente_id STRING, fecha_hora TIMESTAMP, tipo_alerta STRING, motivo STRING, puntuacion_riesgo INT, prioridad STRING, estado STRING, resolucion STRING, fecha_resolucion TIMESTAMP",
}

CLAVES = {
    "oficinas": "oficina_id",
    "gestores": "gestor_id",
    "clientes": "cliente_id",
    "cuentas": "cuenta_id",
    "tarjetas": "tarjeta_id",
    "comercios": "comercio_id",
    "transacciones": "transaccion_id",
    "alertas": "alerta_id",
}

from pyspark.sql.types import StructField, StructType

spark.conf.set("spark.sql.session.timeZone", "UTC")  # las horas de los CSV se guardan tal cual


def no_nulos(df, columnas):
    """Devuelve el DataFrame con las columnas indicadas marcadas como NOT NULL."""
    esquema = StructType([StructField(c.name, c.dataType, c.nullable and c.name not in columnas) for c in df.schema])
    return spark.createDataFrame(df.rdd, esquema)


for tabla, ddl in ESQUEMAS.items():
    df = (spark.read
          .option("header", True)
          .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
          .option("dateFormat", "yyyy-MM-dd")
          .option("quote", '"').option("escape", '"')
          .schema(ddl)
          .csv(f"{CARPETA}/{tabla}.csv"))
    df = no_nulos(df, [CLAVES[tabla]])
    (df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(tabla))
    print(f"{tabla:<15}{spark.table(tabla).count():>9,} filas".replace(",", "."))

## 3. Tablas para la Fabric App

La consola de alertas lee el Lakehouse a través de su punto de conexión SQL de análisis. Para no obligar a la app a cruzar seis tablas en cada consulta, se le preparan dos tablas ya cruzadas:

- `app_alertas`: una fila por alerta, con el cliente, su oficina y gestor, la tarjeta, el comercio y el gasto medio de la tarjeta en los 90 días anteriores.
- `app_movimientos`: las transacciones con el nombre y la categoría del comercio, para enseñar los últimos movimientos de la tarjeta.

La ontología no usa estas tablas: se enlaza a las tablas del banco.

In [ ]:
app_alertas = spark.sql("""
    WITH gasto AS (
        SELECT a.alerta_id, AVG(t.importe_eur) AS gasto_medio_eur
        FROM alertas a
        JOIN transacciones t
          ON t.tarjeta_id = a.tarjeta_id
         AND t.fecha_hora <  a.fecha_hora - INTERVAL 3 HOURS
         AND t.fecha_hora >= a.fecha_hora - INTERVAL 90 DAYS
         AND t.resultado = 'Aprobada'
        GROUP BY a.alerta_id
    )
    SELECT a.alerta_id, a.fecha_hora, a.tipo_alerta, a.motivo, a.puntuacion_riesgo, a.prioridad,
           a.estado, a.resolucion, a.fecha_resolucion,
           t.transaccion_id, t.importe_eur, t.canal, t.pais,
           co.nombre AS comercio, co.categoria AS categoria_comercio, co.municipio AS municipio_comercio,
           ta.tarjeta_id, ta.tipo_tarjeta, ta.tarjeta_enmascarada, ta.estado AS estado_tarjeta,
           c.cliente_id, c.nombre AS cliente, c.segmento, c.provincia AS provincia_cliente,
           o.nombre AS oficina, ge.nombre AS gestor,
           ROUND(gm.gasto_medio_eur, 2) AS gasto_medio_eur
    FROM alertas a
    JOIN transacciones t ON t.transaccion_id = a.transaccion_id
    JOIN comercios co    ON co.comercio_id   = t.comercio_id
    JOIN tarjetas ta     ON ta.tarjeta_id    = a.tarjeta_id
    JOIN clientes c      ON c.cliente_id     = a.cliente_id
    JOIN oficinas o      ON o.oficina_id     = c.oficina_id
    JOIN gestores ge     ON ge.gestor_id     = c.gestor_id
    LEFT JOIN gasto gm   ON gm.alerta_id     = a.alerta_id
""")

app_movimientos = spark.sql("""
    SELECT t.transaccion_id, t.tarjeta_id, t.fecha_hora, t.importe_eur, t.canal, t.pais, t.resultado,
           co.nombre AS comercio, co.categoria AS categoria_comercio, co.municipio AS municipio_comercio,
           co.nivel_riesgo AS riesgo_comercio
    FROM transacciones t
    JOIN comercios co ON co.comercio_id = t.comercio_id
""")

for nombre, df, clave in [("app_alertas", app_alertas, "alerta_id"), ("app_movimientos", app_movimientos, "transaccion_id")]:
    df = no_nulos(df, [clave, "tarjeta_id"])
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(nombre)
    print(f"{nombre:<15}{spark.table(nombre).count():>9,} filas".replace(",", "."))

## 4. Tabla vacía para las decisiones de la app

La ontología enlaza la entidad **Revision** con la tabla `revisiones`. Se crea vacía ahora para poder enlazarla desde el principio; el notebook `02_sincronizar_revisiones` la rellena con las decisiones de la consola de alertas.

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS revisiones (
        revision_id STRING NOT NULL,
        alerta_id STRING,
        decision STRING,
        bloquear_tarjeta BOOLEAN,
        comentario STRING,
        analista_email STRING,
        revisada_en TIMESTAMP
    ) USING DELTA
""")
print(f"revisiones: {spark.table('revisiones').count()} filas")

## 5. Comprobar las cifras del guion

Si alguna comprobación falla, los CSV no son los del repositorio. No vuelvas a generarlos: las respuestas del guion dependen de estas cifras.

In [ ]:
comprobaciones = {
    "Alertas abiertas": ("SELECT COUNT(*) FROM alertas WHERE estado = 'Abierta'", 53),
    "Tarjetas bloqueadas por fraude": ("SELECT COUNT(*) FROM tarjetas WHERE estado = 'Bloqueada'", 37),
    "Alertas en app_alertas": ("SELECT COUNT(*) FROM app_alertas", 949),
    "Transacciones": ("SELECT COUNT(*) FROM transacciones", 161462),
    "Tarjetas en Ponte Alta del 3 al 11/10": ("""
        SELECT COUNT(DISTINCT t.tarjeta_id) FROM transacciones t JOIN comercios c ON c.comercio_id = t.comercio_id
        WHERE c.nombre = 'Gasolineira Ponte Alta' AND t.fecha_hora BETWEEN '2026-10-03' AND '2026-10-11 23:59:59'""", 93),
}
for nombre, (sql, esperado) in comprobaciones.items():
    valor = spark.sql(sql).collect()[0][0]
    print(f"{'OK ' if valor == esperado else 'REVISAR'}  {nombre}: {valor} (esperado {esperado})")

In [ ]:
display(spark.sql("""
    SELECT o.nombre AS oficina, COUNT(*) AS alertas_abiertas
    FROM alertas a JOIN clientes c ON c.cliente_id = a.cliente_id JOIN oficinas o ON o.oficina_id = c.oficina_id
    WHERE a.estado = 'Abierta'
    GROUP BY o.nombre ORDER BY alertas_abiertas DESC LIMIT 5
"""))